## Establecer conexión con SQL

In [25]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import os


In [26]:
# Definir la ruta de la carpeta destino
output_folder = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data"

In [27]:
try:
    connection = mysql.connector.connect(host='212.227.90.6',
                                         database='Equip_15',
                                         user='Equipo15',
                                         password = 'E1q2u3i4p5o15')
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        RRHH = pd.read_sql(f"SELECT * FROM RRHH_07102025", con=connection)
        
       
except Error as e:
    print("Error while connecting to MySQL", e)

Connected to MySQL Server version  8.0.43-0ubuntu0.24.04.1


C:\Users\PC\AppData\Local\Temp\ipykernel_16668\2038054187.py:7: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()
C:\Users\PC\AppData\Local\Temp\ipykernel_16668\2038054187.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  RRHH = pd.read_sql(f"SELECT * FROM RRHH_07102025", con=connection)


## Verificar los tipos iniciales

In [28]:
print("Tipos de variables antes de las conversiones:")
print(RRHH.dtypes)
print("\n")

Tipos de variables antes de las conversiones:
ID                          int64
Reason_absence              int64
Month_absence               int64
Day_week                    int64
Seasons                     int64
Transportation_expense      int64
Distance_Residence_Work     int64
Service_time                int64
Age                         int64
Work_load_Average_day      object
Hit_target                  int64
Disciplinary_failure       object
Education                  object
Son                        object
Social_drinker             object
Social_smoker              object
Pet                        object
Weight                      int64
Height                      int64
Body_mass_index             int64
Absenteeism_hours           int64
dtype: object




In [29]:


# 1. Total de registros
total_registros = len(RRHH)
print("Total de registros:", total_registros)

# 2. Registros completamente únicos (sin duplicados en todas las columnas)
unique_records = RRHH.drop_duplicates()
num_unique_records = len(unique_records)
print("Registros completamente únicos:", num_unique_records)

# 3. Registros con ID único (IDs que aparecen solo una vez)
id_counts = RRHH['ID'].value_counts()
id_unique = id_counts[id_counts == 1]
num_id_unique = len(id_unique)
print("Registros con ID único:", num_id_unique)

# 4. Cantidad de IDs distintos
num_distinct_ids = RRHH['ID'].nunique()
print("IDs distintos:", num_distinct_ids)

# 5. Duplicados exactos (número de registros que son copias exactas de otros)
duplicados_exactos = total_registros - num_unique_records
print("Duplicados exactos (copias extras):", duplicados_exactos)

# 6. ID-Extras (total de registros - IDs distintos)
id_extras = total_registros - num_distinct_ids
print("ID-Extras:", id_extras)

Total de registros: 1110
Registros completamente únicos: 1056
Registros con ID único: 352
IDs distintos: 386
Duplicados exactos (copias extras): 54
ID-Extras: 724


## Cambiar de numérico a categórico

In [30]:
# ID debe permanecer como int64 ya que es un identificador único
for col in ["Month_absence", "Day_week", "Seasons"]:
    RRHH[col] = RRHH[col].astype("object")

## Crear versión numérica y aplicar diccionario para  Reason_Absence

In [31]:
# 1. Crear versión numérica para análisis estadístico
RRHH["Reason_absence_numeric"] = RRHH["Reason_absence"].astype("int64")

# 2. Diccionario CORREGIDO (solo 1-28, sin categoría 0)
reason_absence_dict = {
    0: "No corresponde",
    1: "Enfermedades infecciosas y parasitarias",
    2: "Neoplasias (tumores)",
    3: "Enfermedades de la sangre y órganos hematopoyéticos",
    4: "Enfermedades endocrinas, nutricionales y metabólicas",
    5: "Trastornos mentales y del comportamiento",
    6: "Enfermedades del sistema nervioso",
    7: "Enfermedades del ojo y sus anexos",
    8: "Enfermedades del oído y la apófisis mastoides",
    9: "Enfermedades del sistema circulatorio",
    10: "Enfermedades del sistema respiratorio",
    11: "Enfermedades del sistema digestivo",
    12: "Enfermedades de la piel y tejido subcutáneo",
    13: "Enfermedades del sistema musculoesquelético y tejido conectivo",
    14: "Enfermedades del sistema genitourinario",
    15: "Embarazo, parto y puerperio",
    16: "Condiciones originadas en el período perinatal",
    17: "Malformaciones congénitas y anomalías cromosómicas",
    18: "Síntomas y signos no clasificados",
    19: "Lesiones, envenenamientos y consecuencias de otras causas externas",
    20: "Causas externas de morbilidad y mortalidad",
    21: "Factores que influyen en el estado de salud",
    22: "Paciente en seguimiento",
    23: "Consulta médica",
    24: "Donación de sangre",
    25: "Examen de laboratorio",
    26: "Ausencia injustificada",
    27: "Fisioterapia",
    28: "Consulta dental"
}

# 3. VERIFICAR VALORES ANTES DE MAPEAR
print("🔍 Análisis de valores en Reason_absence antes del mapeo:")
valores_originales = RRHH["Reason_absence_numeric"].value_counts().sort_index()
print(valores_originales)

# 4. Identificar registros problemáticos
valores_fuera_rango = RRHH[~RRHH["Reason_absence_numeric"].between(0, 28)]
if not valores_fuera_rango.empty:
    print(f"⚠️  VALORES FUERA DE RANGO 0-28: {len(valores_fuera_rango)} registros")
    print(f"Valores encontrados: {valores_fuera_rango['Reason_absence_numeric'].unique()}")

🔍 Análisis de valores en Reason_absence antes del mapeo:
Reason_absence_numeric
0      56
1      22
2       3
3       2
4       3
5       6
6      13
7      24
8       7
9       7
10     38
11     46
12     12
13     89
14     31
15      3
16      4
17      2
18     32
19     73
21     11
22     50
23    221
24      5
25     46
26     43
27    105
28    156
Name: count, dtype: int64


## Cambiar de object a numérico

In [32]:
for col in ["Son", "Pet", "Disciplinary_failure", "Social_drinker", "Social_smoker"]:
    # Primero limpiar y luego convertir
    RRHH[col] = RRHH[col].astype(str).str.strip().astype("int64")

## Cambiar separator decimal y tipo a float para Work_load_Average_day

In [33]:
RRHH["Work_load_Average_day"] = (
    RRHH["Work_load_Average_day"]
    .str.replace(",", ".", regex=False) 
    .astype("float64")
)

## Educación - Corrección: Mantener como variable ordinal numérica para análisis de correlación

In [34]:
# Primero crear una columna numérica para análisis
RRHH["Education_numeric"] = RRHH["Education"].astype(str).str.strip().astype("int64")

# Luego crear la columna categórica para visualización
education_map = {
    "1": "High school",
    "2": "Graduate", 
    "3": "Postgraduate",
    "4": "Master/Doctor"
}
RRHH["Education"] = RRHH["Education"].astype(str).str.strip().replace(education_map)

## Eliminar duplicados exactos

In [35]:
RRHH = RRHH.drop_duplicates()
RRHH.shape[0]

1056

## Aplicar diccionario a Reason_absence

In [36]:

# Aplicar el diccionario completo (0-28)
RRHH["Reason_absence"] = RRHH["Reason_absence_numeric"].map(reason_absence_dict)

# Verificación post-mapeo
print("✅ Verificación después del mapeo:")
print(f"Valores nulos en Reason_absence: {RRHH['Reason_absence'].isnull().sum()}")
print(f"Valores únicos en Reason_absence: {RRHH['Reason_absence'].nunique()}")

# Mostrar distribución de razones
print("\n📊 Distribución de Reason_absence:")
print(RRHH["Reason_absence"].value_counts().head(10))

✅ Verificación después del mapeo:
Valores nulos en Reason_absence: 0
Valores únicos en Reason_absence: 28

📊 Distribución de Reason_absence:
Reason_absence
Consulta médica                                                       214
Consulta dental                                                       152
Enfermedades del sistema musculoesquelético y tejido conectivo         85
Fisioterapia                                                           83
Lesiones, envenenamientos y consecuencias de otras causas externas     68
No corresponde                                                         56
Paciente en seguimiento                                                49
Examen de laboratorio                                                  46
Enfermedades del sistema digestivo                                     44
Ausencia injustificada                                                 43
Name: count, dtype: int64


In [37]:
# 1. CONVERSIÓN CON VALIDACIÓN
RRHH["Month_absence_order"] = (
    RRHH["Month_absence"]
    .astype(str)
    .str.strip()
    .apply(lambda x: int(x) if x.isdigit() and 1 <= int(x) <= 12 else None)
)

# 2. IDENTIFICAR REGISTROS PROBLEMÁTICOS (esto debe ejecutarse PRIMERO)
registros_erroneos = RRHH[RRHH["Month_absence_order"].isna()]
print(f"🚫 Se encontraron {len(registros_erroneos)} registros con mes inválido")

if not registros_erroneos.empty:
    print("IDs con mes inválido:", registros_erroneos["ID"].unique().tolist())
    print("Valores originales problemáticos:", registros_erroneos["Month_absence"].unique())
    
    # 3. ANÁLISIS DE IMPACTO (solo si hay registros erróneos)
    print("\n📊 Análisis de registros problemáticos:")
    print(registros_erroneos[["ID", "Month_absence", "Absenteeism_hours"]].describe())
    
    # 4. CALCULAR PORCENTAJE
    filas_antes = RRHH.shape[0]
    porcentaje_problematico = (len(registros_erroneos) / filas_antes) * 100
    print(f"\nPorcentaje de registros problemáticos: {porcentaje_problematico:.2f}%")
    
    if porcentaje_problematico < 5:
        print("✅ Porcentaje bajo - Eliminación recomendada")
    else:
        print("⚠️  Porcentaje alto - Considerar investigación adicional")
    
    

# 6. ACTUALIZAR COLUMNA CATEGÓRICA
month_map = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}
RRHH["Month_absence"] = RRHH["Month_absence_order"].map(month_map)

# 7. VERIFICACIÓN FINAL
print("\n✅ Validación final - Rango de meses:")
print(f"Mínimo: {RRHH['Month_absence_order'].min()}")
print(f"Máximo: {RRHH['Month_absence_order'].max()}")
print("\nDistribución de meses:")
print(RRHH['Month_absence_order'].value_counts().sort_index())

🚫 Se encontraron 7 registros con mes inválido
IDs con mes inválido: [4, 8, 35, 40, 130, 212, 351]
Valores originales problemáticos: [0]

📊 Análisis de registros problemáticos:
               ID  Absenteeism_hours
count    7.000000                7.0
mean   111.428571                0.0
std    129.766273                0.0
min      4.000000                0.0
25%     21.500000                0.0
50%     40.000000                0.0
75%    171.000000                0.0
max    351.000000                0.0

Porcentaje de registros problemáticos: 0.66%
✅ Porcentaje bajo - Eliminación recomendada

✅ Validación final - Rango de meses:
Mínimo: 1.0
Máximo: 12.0

Distribución de meses:
Month_absence_order
1.0      73
2.0     107
3.0     128
4.0      75
5.0      93
6.0      76
7.0      95
8.0      72
9.0      78
10.0     95
11.0     87
12.0     70
Name: count, dtype: int64


## DÍAS DE LA SEMANA - Crear columna de orden y columna con nombre

In [38]:
RRHH["Day_week_order"] = RRHH["Day_week"].astype("int64")

day_map = {
    2: "Lunes", 3: "Martes", 4: "Miercoles", 5: "Jueves", 6: "Viernes"
}
RRHH["Day_week"] = RRHH["Day_week"].astype("object").replace(day_map)

## ESTACIONES DEL AÑO - Crear columna de orden y columna con nombre

In [39]:
RRHH["Seasons_order"] = RRHH["Seasons"].astype("int64")

spanish_season_map = {
    1: "Invierno", 2: "Otono", 3: "Verano", 4: "Primavera"
}
RRHH["Seasons"] = RRHH["Seasons"].astype("object").replace(spanish_season_map)

## Asegurar que todas las variables numéricas continuas estén como float64

In [40]:
numeric_columns = ["Transportation_expense", "Distance_Residence_Work", "Service_time", 
                  "Age", "Hit_target", "Weight", "Height", "Body_mass_index", 
                  "Absenteeism_hours"]

for col in numeric_columns:
    if col in RRHH.columns:
        RRHH[col] = RRHH[col].astype("float64")

## Verificación final de tipos de variables

In [41]:
print("Tipos de variables después de todas las conversiones:")
print(RRHH.dtypes)
print("\n")

Tipos de variables después de todas las conversiones:
ID                           int64
Reason_absence              object
Month_absence               object
Day_week                    object
Seasons                     object
Transportation_expense     float64
Distance_Residence_Work    float64
Service_time               float64
Age                        float64
Work_load_Average_day      float64
Hit_target                 float64
Disciplinary_failure         int64
Education                   object
Son                          int64
Social_drinker               int64
Social_smoker                int64
Pet                          int64
Weight                     float64
Height                     float64
Body_mass_index            float64
Absenteeism_hours          float64
Reason_absence_numeric       int64
Education_numeric            int64
Month_absence_order        float64
Day_week_order               int64
Seasons_order                int64
dtype: object




## Información resumida del DataFrame

In [42]:
print("Resumen del DataFrame:")
print(f"Número de filas: {RRHH.shape[0]}")
print(f"Número de columnas: {RRHH.shape[1]}")
print("\nColumnas por tipo de dato:")
print(RRHH.dtypes.value_counts())

Resumen del DataFrame:
Número de filas: 1056
Número de columnas: 26

Columnas por tipo de dato:
float64    11
int64      10
object      5
Name: count, dtype: int64


In [43]:
# Verifica Reason_absence:
print("🎯 ESTADO FINAL DE REASON_ABSENCE:")
print(f"• Registros totales: {len(RRHH)}")
print(f"• Valores nulos en Reason_absence: {RRHH['Reason_absence'].isnull().sum()}")
print(f"• Valores nulos en Reason_absence_numeric: {RRHH['Reason_absence_numeric'].isnull().sum()}")
print(f"• Rango en Reason_absence_numeric: {RRHH['Reason_absence_numeric'].min()} - {RRHH['Reason_absence_numeric'].max()}")

🎯 ESTADO FINAL DE REASON_ABSENCE:
• Registros totales: 1056
• Valores nulos en Reason_absence: 0
• Valores nulos en Reason_absence_numeric: 0
• Rango en Reason_absence_numeric: 0 - 28


## Exportar a Parquet

In [44]:
try:
    # Asegurar que no hay columnas con tipos problemáticos para Parquet
    # Convertir cualquier object restante a string para evitar problemas
    for col in RRHH.select_dtypes(include=['object']).columns:
        RRHH[col] = RRHH[col].astype("string")
    
    # Construir la ruta completa del archivo Parquet
    parquet_path = os.path.join(output_folder, "RRHH_07102025_clean.parquet")
    
    # Exportar a Parquet
    RRHH.to_parquet(parquet_path, index=False, engine='pyarrow')
    print(f"✅ DataFrame exportado exitosamente a {parquet_path}")
    
except Exception as e:
    print(f"❌ Error al exportar a Parquet: {e}")

# Exportar a CSV (siempre se ejecuta, independientemente del resultado de Parquet)
try:
    # Construir la ruta completa del archivo CSV
    csv_path = os.path.join(output_folder, "RRHH_07102025_clean.csv")
    
    RRHH.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✅ DataFrame exportado exitosamente a {csv_path}")
except Exception as e:
    print(f"❌ Error al exportar a CSV: {e}")

✅ DataFrame exportado exitosamente a C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_07102025_clean.parquet
✅ DataFrame exportado exitosamente a C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_07102025_clean.csv


## Verificación adicional de valores nulos

In [45]:
print("\nValores nulos por columna:")
print(RRHH.isnull().sum())


Valores nulos por columna:
ID                         0
Reason_absence             0
Month_absence              7
Day_week                   0
Seasons                    0
Transportation_expense     0
Distance_Residence_Work    0
Service_time               0
Age                        0
Work_load_Average_day      0
Hit_target                 0
Disciplinary_failure       0
Education                  0
Son                        0
Social_drinker             0
Social_smoker              0
Pet                        0
Weight                     0
Height                     0
Body_mass_index            0
Absenteeism_hours          0
Reason_absence_numeric     0
Education_numeric          0
Month_absence_order        7
Day_week_order             0
Seasons_order              0
dtype: int64


## Cerrar conexión

In [46]:
connection.close()


In [47]:
# Filtrar registros con valores nulos en Month_absence o Month_absence_order
registros_nulos = RRHH[RRHH['Month_absence'].isnull() | RRHH['Month_absence_order'].isnull()]

print("📋 REGISTROS CON VALORES NULOS EN MES:")
print(f"Se encontraron {len(registros_nulos)} registros con problemas\\n")

# Mostrar información detallada de los registros problemáticos
print("🔍 Detalle de registros con valores nulos:")
print(registros_nulos[['ID', 'Month_absence', 'Month_absence_order', 'Reason_absence', 'Absenteeism_hours']].to_string(index=False))

# Mostrar los valores originales que causaron el problema
print(f"\\n📊 Valores originales en 'Month_absence' que causaron nulos:")
valores_originales_problematicos = registros_nulos['Month_absence'].unique()
print(valores_originales_problematicos)

📋 REGISTROS CON VALORES NULOS EN MES:
Se encontraron 7 registros con problemas\n
🔍 Detalle de registros con valores nulos:
 ID Month_absence  Month_absence_order Reason_absence  Absenteeism_hours
  4          <NA>                  NaN No corresponde                0.0
  8          <NA>                  NaN No corresponde                0.0
 35          <NA>                  NaN No corresponde                0.0
 40          <NA>                  NaN No corresponde                0.0
130          <NA>                  NaN No corresponde                0.0
212          <NA>                  NaN No corresponde                0.0
351          <NA>                  NaN No corresponde                0.0
\n📊 Valores originales en 'Month_absence' que causaron nulos:
<StringArray>
[<NA>]
Length: 1, dtype: string
